# Types of Attention

## Prerequisites
Before reading this notebook, Students should know of:

- Attention mechanism
- Sequence to sequence modeling/ encoder-decoder architecture

## Learning Objective

After reading this notebook, students should be able to:

- List and describe different types of attention
- Compare different attention mechanisms
- Explain aligned position computation techniques in local attention.


In the previous chapter, we listed a few issues with typical encoder-decoder architecture and discussed how the attention mechanism fixes those issues. Attention was first motivated by the requirement to remember long term dependencies in a long sequence in neural machine translation(NMT). After the success of the attention mechanism for NMT, it was further explored for its diverse use cases. It was modified to be computationally less intensive and useful for other tasks like Computer Vision, Language Modeling(LM), etc.  This gives rise to different variants/type of attention. In this chapter, we will discuss different types of attention mechanisms.

## Global and Local Attention

Based on the number of encoder's hidden states $h_j^e$ considered while obtaining the context vector $c_i$, the attention mechanism is classified as __Global__ and __Local__ attention.


### Global Attention
Global attention, as the name suggests, considers all the encoder's hidden state $(h_j^e)_{j=1}^{T_x}$ to obtain the context vector $c_i$. The attention mechanism we discussed in the previous chapter is global attention. This consideration of all encoder's hidden states requires a heavy computation for longer sequences, e.g.,  paragraphs or documents. As a solution to this problem, an alternative attention technique called local attention is used.


### Local Attention
Local attention, as the name suggests, considers a small subset of encoder's hidden states $h_j^e$ to obtain the context vector $c_i$. For example, it  considers the hidden states of encoder lying within a window $[-D, D]$ of size/length $2D+1$ to obtain the context vector.


<!-- <img  src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=18T0YPzOfSLiTx9K7RjoKuhNN71Q6awla" alt="global local attention">
<figcaption>Figure 1: Different attention types based on the number of encoder's hidden state considered (i) Global Attention, (ii) Local Attention
</figcaption> -->

<img  src="https://i.postimg.cc/1XcP6YQQ/image.png" alt="global local attention">
<figcaption>Figure 1: Different attention types based on the number of encoder's hidden state considered (i) Global Attention, (ii) Local Attention
</figcaption>


The figure above shows the local and global attention. Notice the number of encoder's state considered while computing the context vector $c_i$, and the alignment weights $\alpha_j$.

Let's discuss local attention in more details.

 <!-- In the figure, $p_i$ is a aligned position which determines the position where the window is kept for $i^{th}$ decoding state. In local attention in above figure, the second, third and fourth word of input sequence lie within the window. So only these three words are considered while computing the attention weights and context vector. We will discuss about aligned position in the next section. -->



Suppose we have a sequence of length $T_x$ and a window of length/size $2D+1$. Then, we will have $T_x-2D$ candidate windows(windows position).  The question is, out of these $T_x-2D$ candidate windows postions, which window to consider at $i^{th}$ decoding stage. Recall that $T_x$ is the length of the input sequence.

The figure below shows an input sequence of length 15. If we consider the window of size 3 ($2D+1=3$), the total number of windows are:

\begin{align*}
 T_x-2D&=  15-2 \\
 &= 13
\end{align*}

Out of these 13 windows, the figure below shows just two windows at two different locations.



<!-- <img  src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1MH2OJ5jDNM4ts1J6R0NAY5Cm24kqG2H6" alt="which window"> -->
<img  src="https://i.postimg.cc/rFD2qmdT/image.png" alt="which window">
<figcaption>Figure 2. Figure showing only two candidate windows each of size 3 at two different locations and the  corresponding texts included within them.
</figcaption>


 If we consider $1^{st}$ window, the hidden vectors corresponding to the word `Local attention considers` will be used to compute the context vector. But if we consider the $2^{nd}$ window, the hidden vectors corresponding the the word `hidden states to` will be used to compute the context vector.




To obtain the appropriate window position, $Luong, M.$ et al., proposes a technique in which we first generate the aligned position represented as $p_i$. The aligned position gives the location in the input sequence where the window is placed at $i^{th}$ decoding state.  For a particular aligned position $p_i$, the window is represented as: $[p_i-D, p_i+D]$. It means the window contains $D$ previous words and $D$ later words from the given position $p_i$ in the input sequence. But how to obtain the aligned position $p_i$ ? There are two techniques to obtain the aligned position:

__1. Monotonic alignment:__ This technique is based on the assumption that source and target sequences are roughly aligned with each other. The aligned position is obtained as:
$$p_i= i$$
where $i$ is the $i^{th}$ decoding state or simply the index word(at current decoding state ) in a target sequence. Since we don't need any additional computation to obtain the aligned position, this technique is quite fast.


__2. Predictive alignment:__
The assumption that source and target sequences are aligned with each other is false. In the predictive alignment technique, the most appropriate aligned position is learned during training. The neural network used to obtain the aligned position is expressed mathematically as:

$$p_i = T_x. \text{sigmoid}(\mathbf{v_p}^{\top}\text{tanh}(\mathbf{W_p}h_{i-1}^d))$$

where, $\mathbf{v_p}$ and $\mathbf{W_p}$ are learnable parameters.

The sigmoid function the above equation ensures that $p_i\in [0, T_x]$.



#### Identify the window for the given value of $p_i$
Using any one of the above two techniques, suppose we obtained $p_i=1$ at $i^{th}$ decoding state. For the input sequence shown in  figure 2, the required window (of size $2D+1$ equal to 3) is:
 $$[p_i-D, p_i+D]=[1-1, 1+1] = [0, 2]$$

This window, $[0, 2]$, turns out to be the same as the first(or left) window shown in figure 2.
So the hidden vector corresponding to index 0 (Local), 1 (attention, and 2(considers) of the input sequence are used to obtain the context vector $c_i$ at $i^{th}$ decoding state.






The overall local attention mechanism with aligned position can be represented as:


<!-- <img  src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1VjCyqKrd5Fh2zhi00Rcm6RO5m1oJVUZA" alt="global local attention">
<figcaption>Figure 3: Local Attention
</figcaption> -->
<img  src="https://i.postimg.cc/L6jKJWVQ/image.png" alt="global local attention">
<figcaption>Figure 3: Local Attention
</figcaption>

## Soft and Hard Attention

Similarly, based on how the alignment weights are distributed among the encoder's hidden state, the attention mechanisms are classified as __Soft__ and __Hard__ attention.





### Soft Attention
In soft attention, the context vector $c_i$ is calculated as a weighted sum of the encoder's hidden states. In other words, the alignment weights $\alpha$ are placed "softly" over all the words in the input sequence. For example, if the alignment weights for the sequence of words with a length of three are $\alpha_{ij} = [0.1, 0.3, 0.6]$, it indicates that 10% of the first word, 30% of the second word, and 60% of the third word in the input sequence are considered while obtaining the context vector $c_i$.
\begin{align*}
c_i&=  \sum_j\alpha_{i, j}*h_j^e   \\
 &=  0.1h_1^e+ 0.3 h_2^e+0.6h_3^e
\end{align*}

 Here, the total probability score of 100% is softly divided into the different words with corresponding hidden vector $h_1^e, h_2^e$, $h_3^e$ in the input sequence. The attention mechanism we discussed in the previous chapter, in fact, is soft attention.


 Note : In $\alpha_{ij}$, the subscript $i$ represents the $i^{th}$ decoding state, and subscript $j$ represent the $j^{th}$ encoder state.

### Hard Attention
In hard attention, the context vector $c_i$ is not calculated as a weighted sum of encoder's hidden states; instead, only one encoder's hidden state is used as a context vector. In other words, the alignment weight is placed "hardly" over a single word in the input sequence.


For example, if the alignment weights for the sequence of words with a length of three are $\alpha_[ij] = [0.1, 0.3, 0.6]$. We use arg max function over alignment weight, and the encoder's hidden state corresponding to the largest value of $\alpha_{ij}$ gives the context vector.

$$\alpha_{i, j}^{max} = \arg\max (\alpha_{ij}) = [0,0,1]$$



\begin{align*}
c_i&=  \sum_j\alpha_{i, j}^{max}*h_j^e   \\
 &=  0*h_1^e +0*h_2^e +1*h_3^e \\
 &= h_3^e
\end{align*}

In practise, rather than using $\arg \max$ function, the hard attention mechanism uses $\alpha$ as a probabilistic mass function to sample over the hidden vectors of encoder $h_j^e$.

The motivation for using hard attention is to reduce the computational requirements. But the problem with hard attention is that the sampling function is __non-differentiable__. We can't use general gradient descent with backpropagation to train the neural network. Instead, we use more advanced and complex techniques like __variance reduction__ or __reinforcement learning__ to train the network. The discussion of these techniques is out of scope.

Apart from Natural Language Processing tasks like NMT, LM, etc., attention mechanisms can be used for computer vision tasks like image generation and image captioning. For instance,
 $Kelvin, X.$ et al., used hard attention for image captioning. You can refer to the paper [here](http://proceedings.mlr.press/v37/xuc15.pdf).

## Key Takeaways

- In global attention, all the encoder's states are considered while computing the context vector, so the name global.

- Local attention mechanism computes the context vector based on the subset of the encoder's hidden state.

- Soft attention places the alignment weight $\alpha$ softly over all the words in the input sequence.

- In hard attention, the context vector is derived from a single hidden state of the encoder.

- Apart from Neural Machine Translation and Language Modeling, attention mechanisms can be used for computer vision tasks.



## Additional Resources

<a name="Paper"></a>
* Paper
 -  Luong, M., et al. “[Effective Approaches to Attention-Based Neural Machine Translation.](http://arxiv.org/abs/1508.04025)” ArXiv:1508.04025 (2015)
